## Test - Ivet


In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


1. Load the sample data


In [26]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

from scipy.stats import linregress

2. Build an ID / date / value table


In [27]:
file_2023 = Path("../data/raw/sample_23.csv")
data_2023 = pd.read_csv(file_2023)

display(data_2023.head())
print("Wide shape:", data_2023.shape)

,ID,2023-01-01,2023-01-02,2023-01-03,2023-01-04,2023-01-05,2023-01-06,2023-01-07,2023-01-08,2023-01-09,...,2023-12-22,2023-12-23,2023-12-24,2023-12-25,2023-12-26,2023-12-27,2023-12-28,2023-12-29,2023-12-30,2023-12-31
0,22,13.482,9.473,10.146,10.978,14.149,11.536,7.767,10.081,9.189,...,8.4100,11.847,7.501,12.0760,12.5340,9.3860,9.5890,7.2150,8.623,12.769
1,42,46.427,49.369,40.441,38.126,40.902,28.853,23.482,42.429,43.268,...,36.6710,46.418,33.754,31.4390,25.3150,25.1880,38.3730,28.3060,32.604,51.493
2,56,9.088,9.300,8.860,13.168,8.341,8.592,14.704,13.383,8.189,...,11.9470,12.490,14.201,16.8190,10.8730,3.4240,0.1420,4.9070,10.655,4.467
3,58,10.040,7.633,11.596,8.036,10.404,6.576,8.617,15.368,8.773,...,14.1510,20.087,13.762,14.6370,21.0030,25.3060,5.2450,13.2220,13.909,16.507
4,64,2.969,2.427,2.018,2.742,2.118,2.879,1.961,2.161,2.256,...,2.5045,2.774,2.018,2.1118,2.1118,2.1118,2.1118,2.1118,2.367,2.729


Wide shape: (17547, 366)


3. Remove series that are all zeros


In [28]:
date_columns = [col for col in data_2023.columns if col != "ID"]

new_data_2023 = data_2023.melt(
    id_vars="ID", value_vars=date_columns, var_name="Date", value_name="Value"
)

display(new_data_2023.head())
print("Long shape:", new_data_2023.shape)

,ID,Date,Value
0,22,2023-01-01,13.482
1,42,2023-01-01,46.427
2,56,2023-01-01,9.088
3,58,2023-01-01,10.040
4,64,2023-01-01,2.969


Long shape: (6404655, 3)


4.Prase dates ig posible

In [29]:
try:
    new_data_2023["Date"] = pd.to_datetime(new_data_2023["Date"])
    print("Date column parsed successfully.")
except Exception:
    print("Could not parse Date column automatically. Keeping as string.")

Date column parsed successfully.


5. Remove all zeros

In [30]:
id_all_zero = new_data_2023.groupby("ID")["Value"].apply(lambda x: (x == 0).all())

print("Number of all-zero IDs:", int(id_all_zero.sum()))

valid_ids = id_all_zero.index[~id_all_zero]
data_2023_filtered = new_data_2023[new_data_2023["ID"].isin(valid_ids)].reset_index(
    drop=True
)

print("Filtered long shape:", data_2023_filtered.shape)
display(data_2023_filtered.head())

Number of all-zero IDs: 154
Filtered long shape: (6348445, 3)


,ID,Date,Value
0,22,2023-01-01,13.482
1,42,2023-01-01,46.427
2,56,2023-01-01,9.088
3,58,2023-01-01,10.040
4,64,2023-01-01,2.969


6. Pivot matrix form

In [ ]:
series_matrix = data_2023_filtered.pivot(index="ID", columns="Date", values="Value")

try:
    series_matrix = series_matrix.sort_index(axis=1)
except Exception:
    pass

display(series_matrix.head())
print("Series matrix shape:", series_matrix.shape)
print("Missing values total:", int(series_matrix.isna().sum().sum()))

Date,2023-01-01,2023-01-02,2023-01-03,2023-01-04,2023-01-05,2023-01-06,2023-01-07,2023-01-08,2023-01-09,2023-01-10,...,2023-12-22,2023-12-23,2023-12-24,2023-12-25,2023-12-26,2023-12-27,2023-12-28,2023-12-29,2023-12-30,2023-12-31
ID,,,,,,,,,,,,,,,,,,,,,
22,13.482,9.473,10.146,10.978,14.149,11.536,7.767,10.081,9.189,14.919,...,8.4100,11.847,7.501,12.0760,12.5340,9.3860,9.5890,7.2150,8.623,12.769
42,46.427,49.369,40.441,38.126,40.902,28.853,23.482,42.429,43.268,36.268,...,36.6710,46.418,33.754,31.4390,25.3150,25.1880,38.3730,28.3060,32.604,51.493
56,9.088,9.300,8.860,13.168,8.341,8.592,14.704,13.383,8.189,8.156,...,11.9470,12.490,14.201,16.8190,10.8730,3.4240,0.1420,4.9070,10.655,4.467
58,10.040,7.633,11.596,8.036,10.404,6.576,8.617,15.368,8.773,6.134,...,14.1510,20.087,13.762,14.6370,21.0030,25.3060,5.2450,13.2220,13.909,16.507
64,2.969,2.427,2.018,2.742,2.118,2.879,1.961,2.161,2.256,3.755,...,2.5045,2.774,2.018,2.1118,2.1118,2.1118,2.1118,2.1118,2.367,2.729


Series matrix shape: (17393, 365)
Missing values total: 0


7. Feture helper functions

In [32]:
def safe_mean(x):
    return float(np.mean(x)) if len(x) > 0 else 0.0


def safe_std(x):
    return float(np.std(x, ddof=0)) if len(x) > 0 else 0.0


def coefficient_of_variation(x):
    m = np.mean(x)
    s = np.std(x, ddof=0)
    return float(s / m) if abs(m) > 1e-8 else 0.0


def autocorr_lag(x, lag):
    x = np.asarray(x, dtype=float)
    if len(x) <= lag:
        return 0.0
    x1 = x[:-lag]
    x2 = x[lag:]
    if np.std(x1) < 1e-8 or np.std(x2) < 1e-8:
        return 0.0
    return float(np.corrcoef(x1, x2)[0, 1])


def trend_slope(x):
    x = np.asarray(x, dtype=float)
    t = np.arange(len(x))
    if np.std(x) < 1e-8:
        return 0.0
    return float(linregress(t, x).slope)


def zero_runs(x):
    """Return lengths of consecutive zero runs."""
    runs = []
    run_len = 0
    for val in x:
        if val == 0:
            run_len += 1
        else:
            if run_len > 0:
                runs.append(run_len)
                run_len = 0
    if run_len > 0:
        runs.append(run_len)
    return runs


def nonzero_stats(x):
    nz = x[x != 0]
    if len(nz) == 0:
        return 0.0, 0.0, 0.0
    return float(np.mean(nz)), float(np.std(nz, ddof=0)), float(np.median(nz))


def rolling_features(x, window=7):
    s = pd.Series(x)
    roll_mean = s.rolling(window=window, min_periods=1).mean()
    roll_std = s.rolling(window=window, min_periods=1).std().fillna(0.0)
    return float(roll_mean.std()), float(roll_std.mean())


def burstiness(x):
    """Simple burstiness index based on std/mean-like behavior."""
    x = np.asarray(x, dtype=float)
    m = np.mean(x)
    s = np.std(x, ddof=0)
    if s + m < 1e-8:
        return 0.0
    return float((s - m) / (s + m))


def peak_features(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 3:
        return 0.0, 0.0
    peaks = []
    for i in range(1, len(x) - 1):
        if x[i] > x[i - 1] and x[i] > x[i + 1]:
            peaks.append(i)
    n_peaks = len(peaks)
    if n_peaks < 2:
        return float(n_peaks), 0.0
    avg_peak_spacing = float(np.mean(np.diff(peaks)))
    return float(n_peaks), avg_peak_spacing


def weekday_profile(x, dates):
    """
    Returns normalized mean consumption for weekday 0..6.
    If dates are not datetime, returns zeros.
    """
    try:
        weekdays = pd.to_datetime(dates).weekday
    except Exception:
        return [0.0] * 7

    x = np.asarray(x, dtype=float)
    overall_mean = np.mean(x)
    if abs(overall_mean) < 1e-8:
        overall_mean = 1.0

    feats = []
    for d in range(7):
        vals = x[weekdays == d]
        feats.append(float(np.mean(vals) / overall_mean) if len(vals) > 0 else 0.0)
    return feats


def seasonal_peak_distribution(x):
    """
    Fraction of top 10% days in first half vs second half of year.
    Very rough seasonal marker.
    """
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return 0.0, 0.0
    threshold = np.quantile(x, 0.9)
    top_idx = np.where(x >= threshold)[0]
    if len(top_idx) == 0:
        return 0.0, 0.0
    first_half = np.mean(top_idx < len(x) / 2)
    second_half = np.mean(top_idx >= len(x) / 2)
    return float(first_half), float(second_half)

8. Extract all custom fetures

In [33]:
def extract_features_from_series_matrix(series_matrix):
    feature_rows = []
    dates = np.array(series_matrix.columns)

    for idx, row in series_matrix.iterrows():
        x = row.to_numpy(dtype=float)

        mean_val = safe_mean(x)
        median_val = float(np.median(x))
        std_val = safe_std(x)
        cv_val = coefficient_of_variation(x)

        p10 = float(np.quantile(x, 0.10))
        p90 = float(np.quantile(x, 0.90))
        p90_p10_ratio = float(p90 / p10) if abs(p10) > 1e-8 else 0.0

        max_val = float(np.max(x))
        max_median_ratio = float(max_val / median_val) if abs(median_val) > 1e-8 else 0.0

        above_p90_fraction = float(np.mean(x > p90)) if len(x) > 0 else 0.0

        slope = trend_slope(x)
        acf1 = autocorr_lag(x, 1)
        acf7 = autocorr_lag(x, 7)
        acf14 = autocorr_lag(x, 14)

        zero_fraction = float(np.mean(x == 0))
        nonzero_fraction = float(np.mean(x != 0))

        zruns = zero_runs(x)
        num_zero_runs = float(len(zruns))
        avg_zero_run_length = float(np.mean(zruns)) if len(zruns) > 0 else 0.0
        max_zero_run_length = float(np.max(zruns)) if len(zruns) > 0 else 0.0

        nonzero_mean, nonzero_std, nonzero_median = nonzero_stats(x)

        rolling7_mean_std, rolling7_std_mean = rolling_features(x, window=7)

        weekday_feats = weekday_profile(x, dates)
        weekday_weekend_gap = float(
            np.mean(weekday_feats[:5]) - np.mean(weekday_feats[5:])
        )

        burst = burstiness(x)
        n_peaks, avg_peak_spacing = peak_features(x)

        top10_first_half, top10_second_half = seasonal_peak_distribution(x)

        feature_rows.append({
            "ID": idx,
            # level / variability
            "mean": mean_val,
            "median": median_val,
            "std": std_val,
            "cv": cv_val,
            "p90_p10_ratio": p90_p10_ratio,
            "max_median_ratio": max_median_ratio,
            "above_p90_fraction": above_p90_fraction,

            # trend / dependence
            "trend_slope": slope,
            "acf1": acf1,
            "acf7": acf7,
            "acf14": acf14,

            # intermittency
            "zero_fraction": zero_fraction,
            "nonzero_fraction": nonzero_fraction,
            "num_zero_runs": num_zero_runs,
            "avg_zero_run_length": avg_zero_run_length,
            "max_zero_run_length": max_zero_run_length,
            "nonzero_mean": nonzero_mean,
            "nonzero_std": nonzero_std,
            "nonzero_median": nonzero_median,

            # rolling
            "rolling7_mean_std": rolling7_mean_std,
            "rolling7_std_mean": rolling7_std_mean,

            # weekday signature
            "weekday_0_norm": weekday_feats[0],
            "weekday_1_norm": weekday_feats[1],
            "weekday_2_norm": weekday_feats[2],
            "weekday_3_norm": weekday_feats[3],
            "weekday_4_norm": weekday_feats[4],
            "weekday_5_norm": weekday_feats[5],
            "weekday_6_norm": weekday_feats[6],
            "weekday_weekend_gap": weekday_weekend_gap,

            # extra custom
            "burstiness": burst,
            "num_peaks": n_peaks,
            "avg_peak_spacing": avg_peak_spacing,
            "top10_first_half": top10_first_half,
            "top10_second_half": top10_second_half,
        })

    return pd.DataFrame(feature_rows)

In [34]:
feature_df = extract_features_from_series_matrix(series_matrix)

display(feature_df.head())
print("Feature table shape:", feature_df.shape)

,ID,mean,median,std,cv,p90_p10_ratio,max_median_ratio,above_p90_fraction,trend_slope,acf1,...,weekday_3_norm,weekday_4_norm,weekday_5_norm,weekday_6_norm,weekday_weekend_gap,burstiness,num_peaks,avg_peak_spacing,top10_first_half,top10_second_half
0,22,11.857921,11.432,3.973155,0.335063,2.367516,2.269419,0.10137,-0.009505,0.589803,...,1.040690,0.914279,0.957808,1.058417,-0.011582,-0.498056,123.0,2.926230,0.945946,0.054054
1,42,14.054819,3.610,18.955428,1.348678,478.927928,22.506371,0.10137,-0.022695,0.907323,...,0.904248,1.034863,1.113167,1.133149,-0.172934,0.148457,106.0,3.428571,0.648649,0.351351
2,56,5.609575,5.739,5.210435,0.928847,94.763504,4.092699,0.10137,-0.016571,0.783508,...,0.922266,0.878647,1.165745,1.098358,-0.185251,-0.036889,113.0,3.232143,0.702703,0.297297
3,58,9.320200,8.057,5.291708,0.567768,3.521269,4.936453,0.10137,0.016987,0.693289,...,0.952755,0.963350,1.069521,1.173497,-0.170780,-0.275699,113.0,3.196429,0.054054,0.945946
4,64,2.438655,2.422,0.514874,0.211130,1.616140,1.816680,0.09863,-0.000220,0.110696,...,0.999158,0.999269,1.014793,0.960737,0.017280,-0.651350,118.0,3.017094,0.631579,0.368421


Feature table shape: (17393, 35)


9. catch22 feutures 

In [35]:
USE_CATCH22 = True

In [36]:
if USE_CATCH22:
    import pycatch22

    catch22_rows = []
    for idx, row in series_matrix.iterrows():
        x = row.to_numpy(dtype=float)
        feats = pycatch22.catch22_all(x)
        row_dict = {"ID": idx}
        for name, value in zip(feats["names"], feats["values"]):
            row_dict[f"catch22_{name}"] = value
        catch22_rows.append(row_dict)

    catch22_df = pd.DataFrame(catch22_rows)
    feature_df = feature_df.merge(catch22_df, on="ID", how="left")

    print("Feature table shape after catch22:", feature_df.shape)

Feature table shape after catch22: (17393, 57)


10. Clena feture matrix

In [37]:
feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

feature_cols = [col for col in feature_df.columns if col != "ID"]

X_features = feature_df[feature_cols].copy()

display(X_features.head())
print(X_features.shape)

,mean,median,std,cv,p90_p10_ratio,max_median_ratio,above_p90_fraction,trend_slope,acf1,acf7,...,catch22_FC_LocalSimple_mean1_tauresrat,catch22_DN_OutlierInclude_p_001_mdrmd,catch22_DN_OutlierInclude_n_001_mdrmd,catch22_SP_Summaries_welch_rect_area_5_1,catch22_SB_BinaryStats_diff_longstretch0,catch22_SB_MotifThree_quantile_hh,catch22_SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1,catch22_SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1,catch22_SP_Summaries_welch_rect_centroid,catch22_FC_LocalSimple_mean3_stderr
0,11.857921,11.432,3.973155,0.335063,2.367516,2.269419,0.10137,-0.009505,0.589803,0.431239,...,0.021277,-0.084932,0.267123,0.659493,4.0,2.115478,0.133333,0.844444,0.171806,0.789591
1,14.054819,3.610,18.955428,1.348678,478.927928,22.506371,0.10137,-0.022695,0.907323,0.769474,...,0.014925,-0.775342,0.071233,0.902894,5.0,1.657275,0.822222,0.866667,0.036816,0.421801
2,5.609575,5.739,5.210435,0.928847,94.763504,4.092699,0.10137,-0.016571,0.783508,0.646670,...,0.010638,-0.284932,0.221918,0.787504,6.0,1.783055,0.133333,0.822222,0.049087,0.621309
3,9.320200,8.057,5.291708,0.567768,3.521269,4.936453,0.10137,0.016987,0.693289,0.620724,...,0.017857,0.852055,0.268493,0.713175,6.0,2.047495,0.200000,0.222222,0.049087,0.705410
4,2.438655,2.422,0.514874,0.211130,1.616140,1.816680,0.09863,-0.000220,0.110696,0.191981,...,0.058824,-0.513699,0.161644,0.332096,5.0,2.162988,0.133333,0.133333,1.362175,1.040229


(17393, 56)


11. scale feutures

In [38]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_features)

print(X_scaled.shape)

(17393, 56)


13. clusturing methids

In [39]:
def evaluate_kmeans(X, k_values):
    results = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X)
        sil = silhouette_score(X, labels)
        results.append({"method": "kmeans", "k": k, "silhouette": sil})
    return pd.DataFrame(results)


def evaluate_gmm(X, k_values):
    results = []
    for k in k_values:
        model = GaussianMixture(n_components=k, random_state=42)
        labels = model.fit_predict(X)
        sil = silhouette_score(X, labels)
        results.append({"method": "gmm", "k": k, "silhouette": sil})
    return pd.DataFrame(results)


def evaluate_agglomerative(X, k_values, linkage="ward"):
    results = []
    for k in k_values:
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        labels = model.fit_predict(X)
        sil = silhouette_score(X, labels)
        results.append({"method": f"agglo_{linkage}", "k": k, "silhouette": sil})
    return pd.DataFrame(results)

In [40]:
k_values = [4, 6, 8, 10, 12, 15, 20]

results_kmeans = evaluate_kmeans(X_scaled, k_values)
results_gmm = evaluate_gmm(X_scaled, k_values)
results_agglo = evaluate_agglomerative(X_scaled, k_values, linkage="ward")

clustering_results = pd.concat(
    [results_kmeans, results_gmm, results_agglo], ignore_index=True
)
clustering_results = clustering_results.sort_values(["silhouette"], ascending=False)

display(clustering_results)

,method,k,silhouette
8,gmm,6,0.988346
0,kmeans,4,0.988345
7,gmm,4,0.988271
14,agglo_ward,4,0.986938
15,agglo_ward,6,0.981591
1,kmeans,6,0.975668
16,agglo_ward,8,0.974065
2,kmeans,8,0.959848
9,gmm,8,0.959799
10,gmm,10,0.959789


14. fit best clusturing midek

In [41]:
best_k = 8

cluster_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = cluster_model.fit_predict(X_scaled)

feature_df["Cluster"] = cluster_labels

display(feature_df.head())
print(feature_df["Cluster"].value_counts().sort_index())

,ID,mean,median,std,cv,p90_p10_ratio,max_median_ratio,above_p90_fraction,trend_slope,acf1,...,catch22_DN_OutlierInclude_p_001_mdrmd,catch22_DN_OutlierInclude_n_001_mdrmd,catch22_SP_Summaries_welch_rect_area_5_1,catch22_SB_BinaryStats_diff_longstretch0,catch22_SB_MotifThree_quantile_hh,catch22_SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1,catch22_SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1,catch22_SP_Summaries_welch_rect_centroid,catch22_FC_LocalSimple_mean3_stderr,Cluster
0,22,11.857921,11.432,3.973155,0.335063,2.367516,2.269419,0.10137,-0.009505,0.589803,...,-0.084932,0.267123,0.659493,4.0,2.115478,0.133333,0.844444,0.171806,0.789591,0
1,42,14.054819,3.610,18.955428,1.348678,478.927928,22.506371,0.10137,-0.022695,0.907323,...,-0.775342,0.071233,0.902894,5.0,1.657275,0.822222,0.866667,0.036816,0.421801,0
2,56,5.609575,5.739,5.210435,0.928847,94.763504,4.092699,0.10137,-0.016571,0.783508,...,-0.284932,0.221918,0.787504,6.0,1.783055,0.133333,0.822222,0.049087,0.621309,0
3,58,9.320200,8.057,5.291708,0.567768,3.521269,4.936453,0.10137,0.016987,0.693289,...,0.852055,0.268493,0.713175,6.0,2.047495,0.200000,0.222222,0.049087,0.705410,0
4,64,2.438655,2.422,0.514874,0.211130,1.616140,1.816680,0.09863,-0.000220,0.110696,...,-0.513699,0.161644,0.332096,5.0,2.162988,0.133333,0.133333,1.362175,1.040229,0


Cluster
0    17340
1        2
2        1
3        5
4       15
5       25
6        3
7        2
Name: count, dtype: int64


cluster 5: high-usage
cluster 4: mostly 0
cluster 6: a lot of spikes
all other clusters are outliars some of the mesures are extriamly big cluster 2 contain the bigest value in the dataset 


15. sumarries

In [42]:
cluster_summary = feature_df.groupby("Cluster")[
    [
        "mean",
        "std",
        "cv",
        "zero_fraction",
        "avg_zero_run_length",
        "acf7",
        "trend_slope",
        "weekday_weekend_gap",
        "burstiness",
        "num_peaks",
    ]
].mean()

display(cluster_summary)

,mean,std,cv,zero_fraction,avg_zero_run_length,acf7,trend_slope,weekday_weekend_gap,burstiness,num_peaks
Cluster,,,,,,,,,,
0,9.214407,4.528200,0.550134,0.017440,1.558407,0.364779,-0.002759,-0.055503,-0.369895,111.858362
1,7.825878,8.634738,1.175960,0.049315,1.642857,0.760207,-0.017920,0.042582,0.063888,101.500000
2,13.240789,13.105098,0.989752,0.063014,1.437500,0.818733,-0.023103,-0.146815,-0.005150,99.000000
3,10.184079,11.880514,1.217036,0.021918,0.927586,0.650625,-0.015130,-0.169105,0.080107,104.200000
4,1.126665,5.996495,3.674290,0.274886,9.753634,0.314297,-0.004305,-0.060708,0.485347,75.666667
5,13.454762,14.050550,1.141857,0.015123,1.056096,0.704177,-0.013761,-0.020851,0.032660,103.280000
6,0.638879,2.034165,6.130081,0.152511,2.171296,0.315757,-0.002800,-0.349928,0.529374,52.666667
7,4.848867,5.775116,1.209581,0.091781,1.180000,0.769340,0.007053,-0.103836,0.090479,115.500000


In [ ]:
cluster_counts = feature_df["Cluster"].value_counts().sort_index()
print(cluster_counts)

Cluster
0    17340
1        2
2        1
3        5
4       15
5       25
6        3
7        2
Name: count, dtype: int64


16. clusters back to ids

In [43]:
id_cluster_map = feature_df[["ID", "Cluster"]].copy()
display(id_cluster_map.head())

,ID,Cluster
0,22,0
1,42,0
2,56,0
3,58,0
4,64,0


17. two satge clusturing

In [44]:
def sparsity_bucket(zero_fraction):
    if zero_fraction < 0.05:
        return "dense"
    elif zero_fraction < 0.25:
        return "moderate_zero"
    else:
        return "high_zero"


feature_df["SparsityGroup"] = feature_df["zero_fraction"].apply(sparsity_bucket)
display(feature_df["SparsityGroup"].value_counts())

SparsityGroup
dense            16648
high_zero          465
moderate_zero      280
Name: count, dtype: int64

In [45]:
two_stage_results = []

for group_name, group_df in feature_df.groupby("SparsityGroup"):
    group_ids = group_df["ID"].values
    group_X = group_df[feature_cols].values

    if len(group_df) < 10:
        continue

    scaler_group = RobustScaler()
    group_X_scaled = scaler_group.fit_transform(group_X)

    k_group = min(5, max(2, len(group_df) // 500))
    k_group = max(2, k_group)

    model_group = KMeans(n_clusters=k_group, random_state=42, n_init=10)
    labels_group = model_group.fit_predict(group_X_scaled)

    tmp = pd.DataFrame(
        {"ID": group_ids, "SparsityGroup": group_name, "SubCluster": labels_group}
    )
    two_stage_results.append(tmp)

two_stage_cluster_df = pd.concat(two_stage_results, ignore_index=True)
display(two_stage_cluster_df.head())
print(two_stage_cluster_df.shape)

,ID,SparsityGroup,SubCluster
0,22,dense,0
1,42,dense,0
2,56,dense,0
3,58,dense,0
4,64,dense,0


(17393, 3)


18. save resukts 

In [46]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

feature_df.to_csv(output_dir / "feature_based_clusters_2023.csv", index=False)
print("Saved feature table with clusters.")

Saved feature table with clusters.
